**Table of contents**<a id='toc0_'></a>    
- 1. [Setup](#toc1_)    
  - 1.1. [Install Dependencies](#toc1_1_)    
  - 1.2. [Import Libraries](#toc1_2_)    
  - 1.3. [Check Runtime / Device](#toc1_3_)    
- 2. [Load Dataset](#toc2_)    
  - 2.1. [Find Project Root](#toc2_1_)    
  - 2.2. [Read Train / Validation Data](#toc2_2_)    
- 3. [Configuration](#toc3_)    
- 4. [Load Qwen Model Locally](#toc4_)    
- 4.5. [Huấn luyện Mô hình Prefix Tuning](#toc4_5_)    
- 5. [Prompting & Inference](#toc5_)    
  - 5.1. [Build Prompt](#toc5_1_)    
  - 5.2. [Generate Summary](#toc5_2_)    
  - 5.3. [Test One Sample](#toc5_3_)    
- 6. [Evaluate Qwen](#toc6_)    
  - 6.1. [Generate Predictions](#toc6_1_)    
  - 6.2. [Compute ROUGE](#toc6_2_)    
  - 6.3. [Optional: Compute BERTScore](#toc6_3_)    
- 7. [Save Outputs](#toc7_)  

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Setup](#toc0_)


## 1.1. <a id='toc1_1_'></a>[Install Dependencies](#toc0_)

In [1]:
# Run this cell once if your environment does not have the required packages.
# In VS Code, make sure the selected kernel is the project's virtual environment.

import sys

!{sys.executable} -m pip install -q -U     pandas pyarrow tqdm     transformers accelerate sentencepiece peft datasets     evaluate rouge-score bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 86.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 86.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.8 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bi

## 1.2. <a id='toc1_2_'></a>[Import Libraries](#toc0_)

In [2]:
import os
import gc
import time
import inspect
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import PrefixTuningConfig, TaskType, get_peft_model

import evaluate

## 1.3. <a id='toc1_3_'></a>[Check Runtime / Device](#toc0_)

In [3]:
print("Python executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
else:
    print("No CUDA GPU detected. Qwen inference on CPU will be very slow.")

Python executable: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU name: Tesla T4
Total VRAM: 14.56 GB


# 2. <a id='toc2_'></a>[Load Dataset](#toc0_)

## 2.1. <a id='toc2_1_'></a>[Find Project Root](#toc0_)

In [4]:
PROJECT_ROOT = Path.cwd()

# If this notebook is opened from the notebooks/ folder, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Current working directory: /kaggle/working
Project root: /kaggle/working


## 2.2. <a id='toc2_2_'></a>[Read Train / Validation Data](#toc0_)

In [5]:
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/minhphuctoiday/datasett")
LOCAL_DATA_DIR = PROJECT_ROOT / "data"

# Prefer Kaggle input paths when running on Kaggle; otherwise use the local project data folder.
DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR

TRAIN_DATA_PATH = DATA_DIR / "train-00000-of-00001.parquet"
VALID_DATA_PATH = DATA_DIR / "valid-00000-of-00001.parquet"

missing_paths = [path for path in [TRAIN_DATA_PATH, VALID_DATA_PATH] if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Cannot find required parquet file(s): "
        + ", ".join(str(path) for path in missing_paths)
    )

print("Using train data file:", TRAIN_DATA_PATH)
print("Using validation data file:", VALID_DATA_PATH)

train_df = pd.read_parquet(TRAIN_DATA_PATH)
valid_df = pd.read_parquet(VALID_DATA_PATH)

# Keep df as the validation dataframe so the existing inference/evaluation cells stay unchanged.
df = valid_df.copy()

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Train columns:", train_df.columns.tolist())
print("Validation columns:", valid_df.columns.tolist())
valid_df.head()

Using train data file: /kaggle/input/datasets/minhphuctoiday/datasettt/train-00000-of-00001.parquet
Using validation data file: /kaggle/input/datasets/minhphuctoiday/datasettt/valid-00000-of-00001.parquet
Train shape: (10775, 2)
Validation shape: (1349, 2)
Train columns: ['article', 'summary']
Validation columns: ['article', 'summary']


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...


# 3. <a id='toc3_'></a>[Configuration](#toc0_)


In [6]:
@dataclass
class EvalConfig:
    # Qwen 1.5B is stronger than 0.5B but needs tighter memory settings.
    model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"

    # Dataset columns
    text_col: str = "article"
    summary_col: str = "summary"

    # Inference settings
    max_input_chars: int = 2500
    max_token_length: int = 1536
    max_new_tokens: int = 128

    # Evaluation settings. Keep small for local testing.
    max_samples: int = 10

    # Deterministic decoding for reproducible evaluation.
    do_sample: bool = False
    num_beams: int = 1
    repetition_penalty: float = 1.1

    # Prefix Tuning settings
    use_sanity_check: bool = True
    num_virtual_tokens: int = 30
    prefix_learning_rate: float = 5e-3
    epochs: int = 3
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    logging_steps: int = 20


config = EvalConfig()
print(config)

EvalConfig(model_name='Qwen/Qwen2.5-1.5B-Instruct', text_col='article', summary_col='summary', max_input_chars=2500, max_token_length=1536, max_new_tokens=128, max_samples=10, do_sample=False, num_beams=1, repetition_penalty=1.1, use_sanity_check=True, num_virtual_tokens=30, prefix_learning_rate=0.005, epochs=3, per_device_train_batch_size=1, gradient_accumulation_steps=4, logging_steps=20)


In [7]:
# Keep only required columns and remove missing rows.
# This prevents NameError problems from undefined TEXT_COL/SUMMARY_COL variables.

TEXT_COL = config.text_col
SUMMARY_COL = config.summary_col

required_cols = [TEXT_COL, SUMMARY_COL]


def clean_summary_dataframe(dataframe: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    missing_cols = [col for col in required_cols if col not in dataframe.columns]

    if missing_cols:
        raise ValueError(
            f"Missing columns in {dataset_name}: {missing_cols}. "
            f"Current columns: {dataframe.columns.tolist()}"
        )

    return dataframe[required_cols].dropna().reset_index(drop=True)


train_df = clean_summary_dataframe(train_df, "train dataset")
valid_df = clean_summary_dataframe(valid_df, "validation dataset")

# Keep df as validation data for the unchanged inference/evaluation cells below.
df = valid_df.copy()

print("Cleaned train shape:", train_df.shape)
print("Cleaned validation shape:", valid_df.shape)
df.head()

Cleaned train shape: (10775, 2)
Cleaned validation shape: (1349, 2)


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...


# 4. <a id='toc4_'></a>[Load Qwen Model Locally](#toc0_)

In [8]:
MODEL_NAME = config.model_name

print("Loading tokenizer:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Use float16 on CUDA to reduce VRAM usage.
# Use float32 on CPU because float16 CPU inference may be unstable/slow.
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading model:", MODEL_NAME)
print("Torch dtype:", torch_dtype)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

# If running on CPU, explicitly move model to CPU.
if not torch.cuda.is_available():
    model = model.to("cpu")

model.config.pad_token_id = tokenizer.pad_token_id
print("Base model loaded successfully.")

peft_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=config.num_virtual_tokens,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

model.eval()
print("Prefix tuning model is ready.")

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading model: Qwen/Qwen2.5-1.5B-Instruct
Torch dtype: torch.float16


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully.
trainable params: 430,080 || all params: 1,544,144,384 || trainable%: 0.0279
Prefix tuning model is ready.


# 4.5. <a id='toc4_5_'></a>[Huấn luyện Mô hình Prefix Tuning](#toc0_)

Phần này chèn thêm luồng huấn luyện Prefix Tuning gồm 3 giai đoạn: chuẩn bị dữ liệu, sanity check nhanh, huấn luyện chính thức với checkpoint tốt nhất.

In [9]:
# Keep Section 4.5 runnable before Section 5.1 while preserving the original build_prompt cell below.
if "build_prompt" not in globals():
    def build_prompt(article: str) -> str:
        # Build instruction prompt for Vietnamese abstractive summarization.
        return f"""
You are a Vietnamese abstractive summarization system.

Summarize the following Vietnamese article in Vietnamese.
Do not copy long sentences directly.
Keep the main ideas, important facts, names, places, numbers, and conclusions.
Write a concise and natural summary.

Article:
{article}

Summary:
""".strip()


SYSTEM_MESSAGE = "You are a helpful assistant specialized in Vietnamese abstractive summarization."


def format_training_example(example):
    article = str(example[TEXT_COL])[:config.max_input_chars]
    summary = str(example[SUMMARY_COL]).strip()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        },
        {
            "role": "user",
            "content": build_prompt(article)
        }
    ]

    # Qwen Instruct training prompt mirrors generate_qwen_summary input formatting.
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    max_prompt_length = max(1, config.max_token_length - config.max_new_tokens)
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_prompt_length
    )["input_ids"]

    target_text = summary
    if tokenizer.eos_token:
        target_text = f"{target_text}{tokenizer.eos_token}"

    target_ids = tokenizer(
        target_text,
        add_special_tokens=False,
        truncation=True,
        max_length=config.max_new_tokens
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


def causal_lm_data_collator(features):
    input_ids = [torch.tensor(feature["input_ids"], dtype=torch.long) for feature in features]
    attention_mask = [torch.tensor(feature["attention_mask"], dtype=torch.long) for feature in features]
    labels = [torch.tensor(feature["labels"], dtype=torch.long) for feature in features]

    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        ),
        "attention_mask": torch.nn.utils.rnn.pad_sequence(
            attention_mask,
            batch_first=True,
            padding_value=0
        ),
        "labels": torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100
        ),
    }


raw_train_dataset = Dataset.from_pandas(train_df[required_cols], preserve_index=False)
raw_eval_dataset = Dataset.from_pandas(valid_df[required_cols], preserve_index=False)

if len(raw_train_dataset) == 0:
    raise ValueError("Train dataset is empty after cleaning.")

if len(raw_eval_dataset) == 0:
    raise ValueError("Validation dataset is empty after cleaning.")

train_dataset = raw_train_dataset.map(
    format_training_example,
    remove_columns=raw_train_dataset.column_names,
    desc="Tokenizing train data"
)

eval_dataset = raw_eval_dataset.map(
    format_training_example,
    remove_columns=raw_eval_dataset.column_names,
    desc="Tokenizing eval data"
)

print("Train samples:", len(train_dataset))
print("Eval samples:", len(eval_dataset))

Tokenizing train data:   0%|          | 0/10775 [00:00<?, ? examples/s]

Tokenizing eval data:   0%|          | 0/1349 [00:00<?, ? examples/s]

Train samples: 10775
Eval samples: 1349


In [10]:
def make_training_args(output_dir, **overrides):
    args = {
        "output_dir": str(output_dir),
        "learning_rate": config.prefix_learning_rate,
        "per_device_train_batch_size": config.per_device_train_batch_size,
        "per_device_eval_batch_size": config.per_device_train_batch_size,
        "gradient_accumulation_steps": config.gradient_accumulation_steps,
        "logging_steps": config.logging_steps,
        "report_to": [],
        "fp16": torch.cuda.is_available(),
        "remove_unused_columns": False,
        "optim": "adamw_torch",
        "save_total_limit": 2,
    }
    args.update(overrides)

    # Compatibility for Transformers versions using eval_strategy instead of evaluation_strategy.
    training_args_params = inspect.signature(TrainingArguments.__init__).parameters
    if "evaluation_strategy" in args and "evaluation_strategy" not in training_args_params:
        if "eval_strategy" in training_args_params:
            args["eval_strategy"] = args.pop("evaluation_strategy")
        else:
            args.pop("evaluation_strategy")

    return TrainingArguments(**args)


def make_trainer(training_args, train_data, eval_data):
    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_data,
        "eval_dataset": eval_data,
        "data_collator": causal_lm_data_collator,
    }

    trainer_params = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    return Trainer(**trainer_kwargs)


if hasattr(model, "config"):
    model.config.use_cache = False

if config.use_sanity_check:
    sanity_train_dataset = train_dataset.select(range(min(32, len(train_dataset))))
    sanity_eval_dataset = eval_dataset.select(range(min(16, len(eval_dataset))))

    sanity_args = make_training_args(
        PROJECT_ROOT / "outputs" / "qwen_prefix_sanity",
        max_steps=10,
        num_train_epochs=1,
        evaluation_strategy="steps",
        eval_steps=5,
        save_strategy="no",
        load_best_model_at_end=False,
    )

    sanity_trainer = make_trainer(sanity_args, sanity_train_dataset, sanity_eval_dataset)
    sanity_result = sanity_trainer.train()

    sanity_loss_history = [
        {"step": entry.get("step"), "loss": entry.get("loss")}
        for entry in sanity_trainer.state.log_history
        if "loss" in entry
    ]

    print("Sanity check metrics:", sanity_result.metrics)
    print("Sanity check training loss history:", sanity_loss_history)
else:
    print("Skipping sanity check because config.use_sanity_check is False.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
5,No log,12.240749
10,No log,9.327881


Sanity check metrics: {'train_runtime': 20.1786, 'train_samples_per_second': 1.982, 'train_steps_per_second': 0.496, 'total_flos': 274007945220096.0, 'train_loss': 11.942495727539063, 'epoch': 1.25}
Sanity check training loss history: []


In [11]:
checkpoint_dir = PROJECT_ROOT / "outputs" / "qwen_prefix_checkpoints"
best_prefix_dir = PROJECT_ROOT / "outputs" / "qwen_prefix_tuning_best"

training_args = make_training_args(
    checkpoint_dir,
    num_train_epochs=config.epochs,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
)

trainer = make_trainer(training_args, train_dataset, eval_dataset)
train_result = trainer.train()

# With load_best_model_at_end=True, trainer.model now holds the best validation-loss checkpoint.
model = trainer.model

if hasattr(model, "config"):
    model.config.use_cache = True

model.eval()

best_prefix_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(best_prefix_dir)
tokenizer.save_pretrained(best_prefix_dir)

print("Training metrics:", train_result.metrics)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)
print("Saved best Prefix Tuning adapter to:", best_prefix_dir)

Epoch,Training Loss,Validation Loss
1,0.741124,0.770597
2,0.734312,0.723027
3,0.686861,0.712050


Training metrics: {'train_runtime': 12154.6766, 'train_samples_per_second': 2.659, 'train_steps_per_second': 0.665, 'total_flos': 2.0731256220085862e+17, 'train_loss': 0.8861850990690946, 'epoch': 3.0}
Best checkpoint: /kaggle/working/outputs/qwen_prefix_checkpoints/checkpoint-8082
Best validation loss: 0.7120500802993774
Saved best Prefix Tuning adapter to: /kaggle/working/outputs/qwen_prefix_tuning_best


# 5. <a id='toc5_'></a>[Prompting & Inference](#toc0_)

## 5.1. <a id='toc5_1_'></a>[Build Prompt](#toc0_)

In [12]:
def build_prompt(article: str) -> str:
    # Build instruction prompt for Vietnamese abstractive summarization.
    return f"""
You are a Vietnamese abstractive summarization system.

Summarize the following Vietnamese article in Vietnamese.
Do not copy long sentences directly.
Keep the main ideas, important facts, names, places, numbers, and conclusions.
Write a concise and natural summary.

Article:
{article}

Summary:
""".strip()

## 5.2. <a id='toc5_2_'></a>[Generate Summary](#toc0_)

In [13]:
def generate_qwen_summary(article: str) -> str:
    # Convert to string and truncate very long article for local testing.
    article = str(article)[:config.max_input_chars]

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant specialized in Vietnamese abstractive summarization."
        },
        {
            "role": "user",
            "content": build_prompt(article)
        }
    ]

    # Qwen Instruct models expect chat template formatting.
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=config.max_token_length
    )

    # Move input tensors to the same device as the model.
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config.max_new_tokens,
            do_sample=config.do_sample,
            num_beams=config.num_beams,
            repetition_penalty=config.repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    # Only decode newly generated tokens, not the original prompt.
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    summary = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return summary.strip()

## 5.3. <a id='toc5_3_'></a>[Test One Sample](#toc0_)


In [14]:
sample = df.iloc[0]

article = sample[TEXT_COL]
reference = sample[SUMMARY_COL]

prediction = generate_qwen_summary(article)

print("ARTICLE:")
print(article[:1000])

print("REFERENCE SUMMARY:")
print(reference)

print("QWEN SUMMARY:")
print(prediction)

ARTICLE:
Giải thưởng công bố gần đây bởi World Travel Awards. Đây là năm thứ hai liên tiếp InterContinental Phu Quoc Long Beach Resort được vinh danh ở hạng mục gia đình trên toàn châu Á. Khu nghỉ dưỡng tọa lạc bên biển Phú Quốc, nổi bật với thiết kế lấy cảm hứng từ đại dương. Khuôn viên rộng rãi với nhiều mảng xanh thiên nhiên đậm chất nhiệt đới. Không gian sảnh lễ tân, phòng nghỉ, villa, nhà hàng... đều được chú trọng để tạo sự hài hòa với biển và cây cối. Du khách có thể chọn nghỉ ngơi tại khu phòng khách sạn rộng rãi, tiện nghi, hoặc những căn hộ, phòng suite, biệt thự cao cấp hướng biển. Mỗi không gian được thiết kế dựa trên tinh thần gắn kết các thành viên trong gia đình. Bên cạnh tiện nghi sang trọng, InterContinental Phu Quoc còn hút khách gia đình nhờ loạt trải nghiệm giải trí, thư giãn đa dạng, phù hợp với mọi độ tuổi. Với trẻ nhỏ, khu Planet Trekker là nơi các bé có thể thoải mái vui chơi, học hỏi từ các đầu sách thiếu nhi, những buổi workshop thủ công... Phụ huynh có thể yê

# 6. <a id='toc6_'></a>[Evaluate Qwen](#toc0_)


## 6.1. <a id='toc6_1_'></a>[Generate Predictions](#toc0_)

In [15]:
test_df = df.head(config.max_samples).copy()

predictions = []
references = []
articles = []

start_time = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating summaries"):
    article = row[TEXT_COL]
    reference = str(row[SUMMARY_COL])

    try:
        pred = generate_qwen_summary(article)
    except RuntimeError as e:
        # Common case: CUDA out of memory.
        print("RuntimeError:", e)
        pred = ""

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    except Exception as e:
        # Keep evaluation running even if one sample fails.
        print("Error:", e)
        pred = ""

    articles.append(article)
    references.append(reference)
    predictions.append(pred)

elapsed = time.time() - start_time
print(f"Generated {len(predictions)} summaries in {elapsed:.2f} seconds")

Generating summaries:   0%|          | 0/10 [00:00<?, ?it/s]

Generated 10 summaries in 66.56 seconds


In [16]:
result_df = pd.DataFrame({
    "article": articles,
    "reference_summary": references,
    "qwen_summary": predictions
})

result_df.head()

,article,reference_summary,qwen_summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...,Việt Nam đã đạt được vị trí 15 trong bảng xếp ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ...",Phú Quốc đã được vinh danh tại hai hạng mục tr...
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...,KKday Vietnam đã hợp tác với Swiss Travel Syst...


## 6.2. <a id='toc6_2_'></a>[Compute ROUGE](#toc0_)

In [17]:
# Remove empty predictions before computing metrics.
valid_pairs = [
    (pred, ref)
    for pred, ref in zip(predictions, references)
    if isinstance(pred, str) and pred.strip()
]

if not valid_pairs:
    raise ValueError("No valid predictions found. Please check model inference errors above.")

valid_predictions, valid_references = zip(*valid_pairs)

rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=list(valid_predictions),
    references=list(valid_references),
    use_stemmer=False
)

rouge_scores

{'rouge1': np.float64(0.712736916420021),
 'rouge2': np.float64(0.44174692474654426),
 'rougeL': np.float64(0.476302607158505),
 'rougeLsum': np.float64(0.47281573482151573)}

## 6.3. <a id='toc6_3_'></a>[Optional: Compute BERTScore](#toc0_)

BERTScore đánh giá mức độ tương đồng ngữ nghĩa tốt hơn ROUGE, đặc biệt khi summary không dùng đúng từ như reference.

In [18]:
# Optional metric. Run this cell if you want semantic similarity score.
# For Vietnamese, lang="vi" is usually acceptable. If it fails, try model_type="xlm-roberta-large".

bertscore = evaluate.load("bertscore")

bert_scores = bertscore.compute(
    predictions=list(valid_predictions),
    references=list(valid_references),
    lang="vi"
)

bert_precision = sum(bert_scores["precision"]) / len(bert_scores["precision"])
bert_recall = sum(bert_scores["recall"]) / len(bert_scores["recall"])
bert_f1 = sum(bert_scores["f1"]) / len(bert_scores["f1"])

print("BERTScore Precision:", bert_precision)
print("BERTScore Recall:", bert_recall)
print("BERTScore F1:", bert_f1)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore Precision: 0.7985496819019318
BERTScore Recall: 0.7704258561134338
BERTScore F1: 0.7840271651744842


# 7. <a id='toc7_'></a>[Save Outputs](#toc0_)

In [19]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

output_file = OUTPUT_DIR / f"qwen_predictions_{config.max_samples}_samples.csv"

result_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saved predictions to:", output_file)

Saved predictions to: /kaggle/working/outputs/qwen_predictions_10_samples.csv


In [20]:
# Save metrics as a small CSV file for report writing.
metrics = {
    "model_name": config.model_name,
    "num_samples": len(valid_predictions),
    "rouge1": rouge_scores.get("rouge1"),
    "rouge2": rouge_scores.get("rouge2"),
    "rougeL": rouge_scores.get("rougeL"),
    "rougeLsum": rouge_scores.get("rougeLsum"),
}

# Add BERTScore if the optional cell was executed.
if "bert_f1" in globals():
    metrics.update({
        "bertscore_precision": bert_precision,
        "bertscore_recall": bert_recall,
        "bertscore_f1": bert_f1,
    })

metrics_df = pd.DataFrame([metrics])
metrics_file = OUTPUT_DIR / f"qwen_metrics_{config.max_samples}_samples.csv"

metrics_df.to_csv(metrics_file, index=False, encoding="utf-8-sig")

print("Saved metrics to:", metrics_file)
metrics_df

Saved metrics to: /kaggle/working/outputs/qwen_metrics_10_samples.csv


,model_name,num_samples,rouge1,rouge2,rougeL,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1
0,Qwen/Qwen2.5-1.5B-Instruct,10,0.712737,0.441747,0.476303,0.472816,0.79855,0.770426,0.784027
